In [2]:
!pip -q install gradio grad-cam

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 61.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision import models, transforms
from PIL import Image

import gradio as gr
import numpy as np
import pandas as pd

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18(weights=None)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)

model.load_state_dict(
    torch.load("pneumonia_resnet18_model (1).pth", map_location=device)
)

model = model.to(device)
model.eval()

classes = ["NORMAL", "PNEUMONIA"]

print("Model loaded successfully.")
print("Using device:", device)

Model loaded successfully.
Using device: cpu


In [17]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Image preprocessing pipeline ready.")

Image preprocessing pipeline ready.


In [18]:
def analyze_xray(image):

    image = image.convert("RGB")

    input_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(input_tensor)
        probabilities = F.softmax(outputs, dim=1)

    normal_prob = probabilities[0][0].item() * 100
    pneumonia_prob = probabilities[0][1].item() * 100

    confidence, predicted = torch.max(probabilities, 1)
    prediction = classes[predicted.item()]
    confidence_score = confidence.item() * 100

    # -------- Grad-CAM --------

    target_layers = [model.layer4[-1]]

    cam = GradCAM(
        model=model,
        target_layers=target_layers
    )

    grayscale_cam = cam(input_tensor=input_tensor)[0]

    resized_image = image.resize((224, 224))
    image_array = np.array(resized_image).astype(np.float32) / 255.0

    gradcam_image = show_cam_on_image(
        image_array,
        grayscale_cam,
        use_rgb=True
    )

    # -------- Analysis Report --------

    report = f"""
# Analysis Report

## Prediction

Prediction: **{prediction}**

Confidence: **{confidence_score:.2f}%**

---

## Class Probability Estimates

| Class | Probability |
|-------|------------:|
| NORMAL | {normal_prob:.2f}% |
| PNEUMONIA | {pneumonia_prob:.2f}% |

---

## Interpretation

Grad-CAM highlights image regions that most influenced the neural network's prediction.

Warm colors (red/yellow) indicate greater influence.

Cool colors (blue/green) indicate lower influence.

Grad-CAM should not be interpreted as identifying the exact location of disease. It is an interpretability technique that visualizes model attention.

---

## Model Information

| Property | Value |
|----------|-------|
| Architecture | ResNet18 |
| Training Method | Transfer Learning |
| Framework | PyTorch |
| Explainability | Grad-CAM |

---

## Disclaimer

This software is provided solely for educational and research purposes.

It is not a medical device and must not be used for clinical diagnosis or patient care.
"""

    probability_table = pd.DataFrame({
        "Class": ["NORMAL", "PNEUMONIA"],
        "Probability (%)": [
            round(normal_prob, 2),
            round(pneumonia_prob, 2)
        ]
    })

    return report, gradcam_image, probability_table


print("Prediction function ready.")

Prediction function ready.


In [19]:
custom_css = """
body, .gradio-container, .markdown, textarea, button, input, label, table {
    font-family: "Times New Roman", Times, serif !important;
}

.gradio-container {
    max-width: 1200px !important;
    margin: auto !important;
}

h1, h2, h3 {
    font-family: "Times New Roman", Times, serif !important;
    font-weight: bold !important;
}

button {
    font-family: "Times New Roman", Times, serif !important;
    font-size: 16px !important;
}
"""

with gr.Blocks(
    title="Explainable Deep Learning for Pneumonia Detection",
    css=custom_css
) as demo:

    gr.Markdown("""
# Explainable Deep Learning for Pneumonia Detection

Research demonstration using a ResNet18 convolutional neural network with Grad-CAM interpretability.

This application is intended solely for educational and research purposes. It is not intended for clinical diagnosis, patient care, or medical decision-making.
""")

    with gr.Row():
        with gr.Column(scale=1):
            input_image = gr.Image(
                type="pil",
                label="Upload Chest Radiograph"
            )

            analyze_button = gr.Button("Run Analysis")

        with gr.Column(scale=1):
            report_output = gr.Markdown(label="Analysis Report")
            gradcam_output = gr.Image(label="Grad-CAM Attention Map")
            probability_output = gr.Dataframe(label="Prediction Probabilities")

    analyze_button.click(
        fn=analyze_xray,
        inputs=input_image,
        outputs=[report_output, gradcam_output, probability_output]
    )

print("Professional interface ready.")

/tmp/ipykernel_2161/264939929.py:22: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Professional interface ready.


In [20]:
demo.launch(
    share=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://371dc9c98dd1fc53e5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [13]:
import os

print(os.getcwd())
print(os.listdir())

/content
['.config', 'pneumonia_resnet18_model (1).pth', 'sample_data']
